# Lab2: 使用 AlexNet 训练 CIFAR-10

实验目标：使用 AlexNet 训练 CIFAR-10

本实验将完成一个完整的图像分类流程，包括：
- 数据加载与预处理
- 模型构建（AlexNet）
- 模型训练
- 模型测试与评估

我们使用 CIFAR-10 数据集。

In [41]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import os

# 设置随机种子以确保可重复性
torch.manual_seed(42)

# 定义设备为 GPU（如果可用），否则使用 CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


## 数据预处理

为了提升模型的泛化能力，我们对训练数据进行增强：
- 随机裁剪（RandomCrop）
- 随机翻转（HorizontalFlip）

测试集不做增强，只做标准化处理。

In [42]:
transform_train = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

## 加载数据集

解压数据集，并查看数据集内容

In [43]:
import os
import zipfile
import torchvision
from torch.utils.data import DataLoader


# 解压 CIFAR10.zip
zip_path = './CIFAR10.zip'
data_root = './CIFAR10'

extract_flag = os.path.join(data_root, 'train')

if not os.path.exists(data_root):
    os.makedirs(data_root)

if not os.path.exists(extract_flag):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(data_root)

    print("解压完成")
else:
    print("数据已存在，跳过解压")


train_dir = os.path.join(data_root, 'train')
val_dir = os.path.join(data_root, 'val')
test_dir = os.path.join(data_root, 'test')


if not os.path.isdir(train_dir):
    raise FileNotFoundError(f'未找到训练集目录: {train_dir}')

# ImageFolder 需要目录结构为 split/class_x/xxx.jpg
# 若 test 不是按类别分目录，则回退到 val

def has_class_subdirs(folder_path):
    if not os.path.isdir(folder_path):
        return False
    for name in os.listdir(folder_path):
        if os.path.isdir(os.path.join(folder_path, name)):
            return True
    return False

if has_class_subdirs(test_dir):
    eval_dir = test_dir
elif has_class_subdirs(val_dir):
    eval_dir = val_dir
else:
    raise FileNotFoundError(
        f'test/val 都不符合 ImageFolder 目录结构，请检查: {test_dir} 和 {val_dir}'
    )

# 下面四行分别构建训练集/训练加载器与测试集/测试加载器
# 提示1：ImageFolder 需要目录结构为 `split/class_x/xxx.jpg`，并且要配合对应的 transform
# 提示2：DataLoader 的常见设置：训练集 `shuffle=True`，测试集 `shuffle=False`
# TODO: trainset（用 train_dir + transform_train 构造 ImageFolder）
trainset = torchvision.datasets.ImageFolder(root=train_dir, transform=transform_train)
# TODO: trainloader（用 trainset 构造 DataLoader；batch_size=128，shuffle=True，num_workers=2）
trainloader = DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)

# TODO: testset（用 eval_dir + transform_test 构造 ImageFolder）
testset = torchvision.datasets.ImageFolder(root=eval_dir, transform=transform_test)
# TODO: testloader（用 testset 构造 DataLoader；batch_size=100，shuffle=False，num_workers=2）
testloader = DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)

print('使用本地数据目录:', data_root)
print('评估集目录:', eval_dir)
print('训练样本数:', len(trainset), '评估样本数:', len(testset))

数据已存在，跳过解压
使用本地数据目录: ./CIFAR10
评估集目录: ./CIFAR10\val
训练样本数: 3306 评估样本数: 364


## 模型构建：AlexNet

AlexNet 是经典的卷积神经网络，包含：
- 多层卷积层（提取特征）
- 池化层（降维）
- 全连接层（分类）


In [44]:
# 定义 AlexNet 模型
class AlexNet(nn.Module):
    def __init__(self, num_classes=10):
        super(AlexNet, self).__init__()
        # 卷积特征提取
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True), # 最大池化
            nn.MaxPool2d(kernel_size=2),
            
            # TODO: 这一层输入是 64，输出是 192，卷积核 3×3，padding=1
            nn.Conv2d(64, 192, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            # TODO: 2×2 最大池化，用于下采样
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),
        )
        # 平均池化
        self.avgpool = nn.AdaptiveAvgPool2d((6, 6))
        # 全连接分类
        self.classifier = nn.Sequential(
            nn.Dropout(), # 防止过拟合
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes),
        )

    def forward(self, x):
        # 数据流
        x = self.features(x)
        # TODO: 自适应平均池化
        x = self.avgpool(x)
        x = torch.flatten(x, 1) # 展平
        # TODO: 全连接分类
        x = self.classifier(x)
        return x

## 初始化模型与优化器
- 损失函数：交叉熵（分类问题常用）
- 优化器：SGD

In [45]:
# 初始化模型、损失函数和优化器
net = AlexNet(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40)

## 训练函数

训练流程包括：
1. 前向传播（forward）
2. 计算损失（loss）
3. 反向传播（backward）
4. 参数更新（optimizer.step）

In [46]:
def train(epoch):
    net.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (inputs, targets) in enumerate(trainloader):
        inputs, targets = inputs.to(device), targets.to(device)
        
        # TODO: 前向传播（输入模型得到输出）
        outputs = net(inputs)
        # TODO:计算损失函数
        loss = criterion(outputs, targets)
        # TODO: 反向传播 + 参数更新（三步）
        optimizer.zero_grad()         # 清空梯度
        loss.backward()        # 计算梯度
        optimizer.step()         # 更新参数
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
    
    print(f"Epoch {epoch+1} - Train Loss: {running_loss/len(trainloader):.4f}, Accuracy: {100.*correct/total:.2f}%")

## 测试函数

测试阶段：
- 不进行梯度计算（节省资源）
- 只评估模型性能

In [47]:
def test(epoch):
    global best_acc
    net.eval()
    test_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(testloader):
            inputs, targets = inputs.to(device), targets.to(device)
            
            outputs = net(inputs)
            loss = criterion(outputs, targets)
            
            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    
    acc = 100.*correct/total
    print(f"Epoch {epoch+1} - Test Loss: {test_loss/len(testloader):.4f}, Accuracy: {acc:.2f}%")

    if acc > best_acc:
        print('Saving..')
        state = {
            'net': net.state_dict(),
            'acc': acc,
            'epoch': epoch,
        }
        if not os.path.exists('./checkpoint'):
            os.makedirs('./checkpoint')

        torch.save(state, './checkpoint/ckpt.pth')
        best_acc = acc

## 开始训练

设置训练轮数（epoch），循环训练并测试模型。

In [48]:
best_acc = 0
num_epochs = 20

for epoch in range(num_epochs):
    train(epoch)
    test(epoch)
    scheduler.step()

print("Training complete.")

Epoch 1 - Train Loss: 2.1222, Accuracy: 21.99%
Epoch 1 - Test Loss: 1.6181, Accuracy: 24.45%
Saving..
Epoch 2 - Train Loss: 1.7043, Accuracy: 19.84%
Epoch 2 - Test Loss: 1.6170, Accuracy: 24.45%
Epoch 3 - Train Loss: 1.6219, Accuracy: 22.32%
Epoch 3 - Test Loss: 1.5927, Accuracy: 21.70%
Epoch 4 - Train Loss: 1.6119, Accuracy: 24.32%
Epoch 4 - Test Loss: 1.5989, Accuracy: 31.04%
Saving..
Epoch 5 - Train Loss: 1.5938, Accuracy: 27.34%
Epoch 5 - Test Loss: 1.5740, Accuracy: 21.70%
Epoch 6 - Train Loss: 1.5860, Accuracy: 29.43%
Epoch 6 - Test Loss: 1.5233, Accuracy: 43.41%
Saving..
Epoch 7 - Train Loss: 1.4524, Accuracy: 36.57%
Epoch 7 - Test Loss: 1.3753, Accuracy: 43.41%
Epoch 8 - Train Loss: 1.2998, Accuracy: 42.86%
Epoch 8 - Test Loss: 1.2240, Accuracy: 45.88%
Saving..
Epoch 9 - Train Loss: 1.2150, Accuracy: 46.34%
Epoch 9 - Test Loss: 1.1992, Accuracy: 49.45%
Saving..
Epoch 10 - Train Loss: 1.1846, Accuracy: 48.43%
Epoch 10 - Test Loss: 1.1298, Accuracy: 49.73%
Saving..
Epoch 11 - Tra